# Notes

# Setup

In [1]:
print('hello')

hello


In [2]:
%load_ext autoreload
%autoreload 2

import os
import re
from scipy import stats

import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import numpy as np
import rasterio

import config
import hatfield_funcs as hf
import hatfield_funcs.helpers as hfh

from pathlib import Path
import sys

project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from my_extra_to_hatfield_funcs.io_funcs import (
    read_xlsx,
    read_df_from_csv,
    write_df_to_csv,
    read_geojson,
    write_geojson,
)

from biofilm_data.io import (
    load_full_station_dataset,
    load_calibration_curves,
)

from biofilm_data.plot import (
    plot_r_vs_s_comparison,
)

from biofilm_data.calibration_curve import (
    fit_linear_calibration,
    plot_calibration_fit,
    mean_response_with_ci,
)

plt.style.use("default")
%config InlineBackend.figure_format = 'retina'

In [3]:
if os.getenv("GA_STORAGE_ACCOUNT_NAME") is None:
    raise RuntimeError(
        "This notebook is configured for Azure cloud storage. "
        "Set GA_STORAGE_ACCOUNT_NAME before running it."
    )

hf.config.configure_azure()
hf.config.configure_data_root_path(config.CONTAINER_NAME)

# Parameters

# Read Data

In [4]:
df_all_R = load_full_station_dataset(
    config.CONTAINER_FILE_PATHS["full_station_dataset_R_geojson"],
    r_only=True
)

display(df_all_R.head())

# df_all = load_full_station_dataset(
#     config.CONTAINER_FILE_PATHS["full_station_dataset_geojson"], r_only=False
# )

# display(df_all.head())

date  elevation             low_tide_time  \
site      round station_id                                                   
brunswick 1     B1         2026-05-03    3.46975 2026-05-03 13:10:00-07:00   
                B10        2026-05-01    3.37800 2026-05-01 12:11:00-07:00   
                B11        2026-05-01    3.20575 2026-05-01 12:11:00-07:00   
                B12        2026-05-01    3.04725 2026-05-01 12:11:00-07:00   
                B13        2026-05-01    2.85350 2026-05-01 12:11:00-07:00   

                            low_tide_level            emergence_time  \
site      round station_id                                             
brunswick 1     B1                   0.862 2026-05-03 07:28:43-07:00   
                B10                  1.053 2026-05-01 07:07:45-07:00   
                B11                  1.053 2026-05-01 07:28:48-07:00   
                B12                  1.053 2026-05-01 07:46:45-07:00   
                B13                  1.053 2026-05-01 08:07:33-07:00   

                                    submergence_time  exposure_duration_hours  \
site      round station_id                                                      
brunswick 1     B1         2026-05-03 18:23:52-07:00                10.919213   
                B10        2026-05-01 17:06:07-07:00                 9.972685   
                B11        2026-05-01 16:48:04-07:00                 9.321157   
                B12        2026-05-01 16:32:26-07:00                 8.761250   
                B13        2026-05-01 16:13:57-07:00                 8.106742   

                           status          acquisition_time  \
site      round station_id                                    
brunswick 1     B1             ok 2026-05-03 10:30:20-07:00   
                B10            ok 2026-05-01 14:05:55-07:00   
                B11            ok 2026-05-01 10:34:02-07:00   
                B12            ok 2026-05-01 13:26:58-07:00   
                B13            ok 2026-05-01 11:25:54-07:00   

                           emergence_acquisition_time_difference  ...  \
site      round station_id                                        ...   
brunswick 1     B1                         P0DT3H1M37.149275336S  ...   
                B10                        P0DT6H58M9.700157357S  ...   
                B11                        P0DT3H5M13.316590501S  ...   
                B12                       P0DT5H40M12.650216295S  ...   
                B13                       P0DT3H18M20.949336115S  ...   

                            imagery_ndvi_mean_DLS_res_full  \
site      round station_id                                   
brunswick 1     B1                                0.128217   
                B10                               0.129499   
                B11                               0.124070   
                B12                               0.083047   
                B13                               0.081480   

                            imagery_ndvi_mean_DLS_res_1m  \
site      round station_id                                 
brunswick 1     B1                              0.129182   
                B10                             0.126515   
                B11                             0.119498   
                B12                             0.080205   
                B13                             0.080520   

                            imagery_ndvi_mean_DLS_res_2m  \
site      round station_id                                 
brunswick 1     B1                              0.128699   
                B10                             0.118512   
                B11                             0.119826   
                B12                             0.080439   
                B13                             0.081106   

                            imagery_ndvi_mean_DLS_res_5m  \
site      round station_id                                 
brunswick 1     B1                              0.133224   
  

# END